## Part 2: Prepare ELSET Data
#### 1. Acquire the latest General Perturbation (gp) Data in form of Orbit Mean-Element Messages (OMMs) from [space-track.org](https://www.space-track.org/)

In [2]:
# Reauthenticate with Space-Track.org to avoid having to run first cell again

# Space-Track.org API authentication 
from spacetrack import SpaceTrackClient
import pandas as pd

# read Space-Track.org credentials from external file (username in line 1, password in line 2)
with open('../DS_SpaceTrackCredentials.txt', 'r') as file:
    credentials = file.read().splitlines()

username = credentials[0]
password = credentials[1]

# create a SpaceTrackClient instance
st = SpaceTrackClient(username, password) 

try:
    # Try fetching a single SATCAT entry to verify authentication
    test_data = st.gp(limit=1, format='json')
    print("Successfully authenticated with Space-Track.org!")
except Exception as e:
    print(f"Failed to authenticate with Space-Track.org: {e}")

print("Test data: " + test_data)

Successfully authenticated with Space-Track.org!
Test data: [{"CCSDS_OMM_VERS":"3.0","COMMENT":"GENERATED VIA SPACE-TRACK.ORG API","CREATION_DATE":"2025-02-13T17:23:10","ORIGINATOR":"18 SPCS","OBJECT_NAME":"TBA - TO BE ASSIGNED","OBJECT_ID":"UNKNOWN","CENTER_NAME":"EARTH","REF_FRAME":"TEME","TIME_SYSTEM":"UTC","MEAN_ELEMENT_THEORY":"SGP4","EPOCH":"2025-02-13T10:26:29.120928","MEAN_MOTION":"15.35778967","ECCENTRICITY":"0.00904350","INCLINATION":"87.9732","RA_OF_ASC_NODE":"300.7112","ARG_OF_PERICENTER":"59.9341","MEAN_ANOMALY":"301.0842","EPHEMERIS_TYPE":"0","CLASSIFICATION_TYPE":"U","NORAD_CAT_ID":"270446","ELEMENT_SET_NO":"999","REV_AT_EPOCH":"2587","BSTAR":"0.00864688340000","MEAN_MOTION_DOT":"0.00357922","MEAN_MOTION_DDOT":"0.0000000000000","SEMIMAJOR_AXIS":"6836.745","PERIOD":"93.763","APOAPSIS":"520.438","PERIAPSIS":"396.782","OBJECT_TYPE":"UNKNOWN","RCS_SIZE":null,"COUNTRY_CODE":null,"LAUNCH_DATE":null,"SITE":null,"DECAY_DATE":null,"FILE":"4639754","GP_ID":"280631293","TLE_LINE0":

In [3]:
# Read the NORAD IDs from the file
norad_ids = pd.read_csv("../Data/DS_NoradIDs.csv")
print(f"Objects: {len(norad_ids)}")

Objects: 25722


In [4]:
# Split the IDs into chunks of 500 to avoid violating the API query restrictions
norad_ids_chunks = []
chunk_size = 500
for i in range(0, len(norad_ids), chunk_size):
    chunk = norad_ids[i:i + chunk_size]
    norad_ids_chunks.append(chunk)

# show number and size of chunks
print(f"Number of chunks: {len(norad_ids_chunks)}")
for i in range(len(norad_ids_chunks)):
    print(f"Chunk {i+1}: {len(norad_ids_chunks[i])}")

Number of chunks: 52
Chunk 1: 500
Chunk 2: 500
Chunk 3: 500
Chunk 4: 500
Chunk 5: 500
Chunk 6: 500
Chunk 7: 500
Chunk 8: 500
Chunk 9: 500
Chunk 10: 500
Chunk 11: 500
Chunk 12: 500
Chunk 13: 500
Chunk 14: 500
Chunk 15: 500
Chunk 16: 500
Chunk 17: 500
Chunk 18: 500
Chunk 19: 500
Chunk 20: 500
Chunk 21: 500
Chunk 22: 500
Chunk 23: 500
Chunk 24: 500
Chunk 25: 500
Chunk 26: 500
Chunk 27: 500
Chunk 28: 500
Chunk 29: 500
Chunk 30: 500
Chunk 31: 500
Chunk 32: 500
Chunk 33: 500
Chunk 34: 500
Chunk 35: 500
Chunk 36: 500
Chunk 37: 500
Chunk 38: 500
Chunk 39: 500
Chunk 40: 500
Chunk 41: 500
Chunk 42: 500
Chunk 43: 500
Chunk 44: 500
Chunk 45: 500
Chunk 46: 500
Chunk 47: 500
Chunk 48: 500
Chunk 49: 500
Chunk 50: 500
Chunk 51: 500
Chunk 52: 222


In [5]:
# Convert each chunk to a string to avoid violating the API query limit
chunk_strings=[]
for chunk in norad_ids_chunks:
    if not chunk.empty:
        chunk_string = ",".join(chunk["NORAD_CAT_ID"].astype(str))
        chunk_strings.append(chunk_string)


In [6]:
# Retreive OMM data for each chunk
import time # Import the time module to add delays between API requests
import json # Import the json module to parse the API response

omm_data_chunks = []
omm_data_received = 0

for i, chunk_string in enumerate(chunk_strings):
    try:
        # retreive omm data
        omm_data = st.gp(norad_cat_id=chunk_string, format='json')
        omm_data_chunks.append(omm_data)

        # Parse the json response to count the elements
        omm_data_json = json.loads(omm_data)
        json_element_count = len(omm_data_json)

        print(f"Retrieved OMM data for chunk {i+1} with {len(chunk_string.split(','))} IDs requested and {json_element_count} elements returned")
        omm_data_received += json_element_count
    except Exception as e:
        print(f"Error retrieving OMM data for chunk {i+1}: {e}")

    # Add a delay of 5 seconds between API requests to avoid rate limiting
    if i < len(chunk_strings) - 1:
        print("Waiting for 5 seconds before the next request...")
        time.sleep(5)

print(f"Total OMMs requested: {len(norad_ids)}")
print(f"Total OMM data elements received: {omm_data_received}")
print("Missing: " + str(len(norad_ids) - omm_data_received))

Retrieved OMM data for chunk 1 with 500 IDs requested and 474 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 2 with 500 IDs requested and 368 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 3 with 500 IDs requested and 409 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 4 with 500 IDs requested and 480 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 5 with 500 IDs requested and 490 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 6 with 500 IDs requested and 402 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 7 with 500 IDs requested and 427 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 8 with 500 IDs requested and 255 elements returned
Waiting for 5 sec

In [7]:
# Save the OMM data chunks to a dataframe
dataframes = [pd.DataFrame(json.loads(chunk)) for chunk in omm_data_chunks]
OMMs_df = pd.concat(dataframes, ignore_index=True)

# show all columns of the df
pd.set_option('display.max_columns', None)
OMMs_df

,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,COUNTRY_CODE,LAUNCH_DATE,SITE,DECAY_DATE,FILE,GP_ID,TLE_LINE0,TLE_LINE1,TLE_LINE2
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:26,18 SPCS,THOR ABLESTAR R/B,1960-003A,EARTH,TEME,UTC,SGP4,1961-08-16T14:24:53.774496,16.29358349,0.00000000,51.2500,356.4892,331.9473,28.0526,0,U,30,999,7590,0.00000000000000,0.02125614,0.0000000000000,6572.405,88.378,194.270,194.270,ROCKET BODY,None,None,None,None,1961-08-18,33907,8672722,0 THOR ABLESTAR R/B,1 00030U 60003 A 61228.60062239 .02125614 +0...,2 00030 051.2500 356.4892 0000000 331.9473 028...
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,THOR ABLESTAR DEB,1960-003C,EARTH,TEME,UTC,SGP4,1960-10-06T07:59:06.496512,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,0,U,33,999,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,None,1960-07-17,33906,8671947,0 THOR ABLESTAR DEB,1 00033U 60003 C 60280.33271408 .00001932 +0...,2 00033 051.2879 136.0668 0242757 196.7800 163...
2,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,SL-3 R/B,1960-005B,EARTH,TEME,UTC,SGP4,1960-07-16T04:00:11.150208,16.27523673,0.00000000,64.8899,44.6976,36.3819,323.6180,0,U,35,999,990,0.00000000000000,0.03271057,0.0000000000000,6577.344,88.477,199.209,199.209,ROCKET BODY,None,None,None,None,1960-07-17,33906,8671988,0 SL-3 R/B,1 00035U 60005 B 60198.16679572 .03271057 +0...,2 00035 064.8899 044.6976 0000000 036.3819 323...
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:26,18 SPCS,SPUTNIK 4 DEB,1960-005D,EARTH,TEME,UTC,SGP4,1961-03-15T15:48:42.802271,15.49800338,0.01984030,64.8899,343.2411,333.4201,27.0840,0,U,37,999,4690,0.00000000000000,0.00039007,0.0000000000000,6795.450,92.915,552.139,282.492,DEBRIS,None,None,None,None,1961-06-30,33907,8672884,0 SPUTNIK 4 DEB,1 00037U 60005 D 61074.65882873 .00039007 +0...,2 00037 064.8899 343.2411 0198403 333.4201 027...
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,SPUTNIK 4 DEB,1960-005E,EARTH,TEME,UTC,SGP4,1960-08-19T16:34:33.722976,15.88909013,0.00342130,64.8899,302.1218,49.4823,310.6691,0,U,38,999,1500,0.00000000000000,0.00610805,0.0000000000000,6683.481,90.628,328.212,282.480,DEBRIS,None,None,None,None,1960-08-20,33906,8672029,0 SPUTNIK 4 DEB,1 00038U 60005 E 60232.69066809 .00610805 +0...,2 00038 064.8899 302.1218 0034213 049.4823 310...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24620,3.0,GENERATED VIA SPACE-TRACK.ORG API,2024-12-12T06:56:16,18 SPCS,CZ-6A DEB,2024-140XY,EARTH,TEME,UTC,SGP4,2024-12-11T23:05:15.774432,15.90490446,0.00187510,89.2070,338.9347,209.0449,150.9793,0,U,61407,999,1395,0.03161600000000,0.07121792,0.0914950000000,6679.046,90.538,313.435,288.388,DEBRIS,SMALL,PRC,2024-08-06,TSC,2024-12-17,4579024,275424010,0 CZ-6A DEB,1 61407U 24140XY 24346.96198813 .07121792 9...,2 61407 89.2070 338.9347 0018751 209.0449 150...
24621,3.0,GENERATED VIA SPACE-TRACK.ORG API,2024-11-13T18:05:02,18 SPCS,CZ-6A DEB,2024-140ZF,EARTH,TEME,UTC,SGP4,2024-11-13T06:57:45.539424,16.26800607,0.00303320,90.0847,353.4507,44.4626,315.9104,0,U,61438,999,1498,0.00486610000000,0.17891631,-0.0000005426100,6579.289,88.517,221.110,181.198,DEBRIS,SMALL,PRC,2024-08-06,TSC,2024-11-16,4552726,273063700,0 CZ-6A DEB,1 61438U 24140ZF 24318.29011041 .17891631 -5...,2 61438 90.0847 353.4507 0030332 44.4626 315...
24622,3.0,GENERATED VIA SPACE-TRACK.ORG API,2024-11-10T13:46:22,18 SPCS,CZ-2D R/B,2024-177B,EARTH,TEME,UTC,SGP4,2024-11-10T05:25:56.203680,16.37905997,0.0

#### 2. Inspect data to see which columns are unneccecary 

In [8]:
# inspect certain columns to see if they can be dropped
print(OMMs_df['ORIGINATOR'].unique())
print(OMMs_df['CENTER_NAME'].unique())
print(OMMs_df['REF_FRAME'].unique())
print(OMMs_df['TIME_SYSTEM'].unique())
print(OMMs_df['MEAN_ELEMENT_THEORY'].unique())
print(OMMs_df['EPHEMERIS_TYPE'].unique())
print(OMMs_df['CLASSIFICATION_TYPE'].unique())
print(OMMs_df['ELEMENT_SET_NO'].unique())

['18 SPCS']
['EARTH']
['TEME']
['UTC']
['SGP4']
['0']
['U']
['999']


In [9]:
# drop not needed columns
OMMs_df.drop(columns=['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'ELEMENT_SET_NO', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2', 'COUNTRY_CODE'], inplace=True)
OMMs_df

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,LAUNCH_DATE,SITE,DECAY_DATE
0,THOR ABLESTAR R/B,1960-003A,1961-08-16T14:24:53.774496,16.29358349,0.00000000,51.2500,356.4892,331.9473,28.0526,30,7590,0.00000000000000,0.02125614,0.0000000000000,6572.405,88.378,194.270,194.270,ROCKET BODY,None,None,None,1961-08-18
1,THOR ABLESTAR DEB,1960-003C,1960-10-06T07:59:06.496512,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,33,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,1960-07-17
2,SL-3 R/B,1960-005B,1960-07-16T04:00:11.150208,16.27523673,0.00000000,64.8899,44.6976,36.3819,323.6180,35,990,0.00000000000000,0.03271057,0.0000000000000,6577.344,88.477,199.209,199.209,ROCKET BODY,None,None,None,1960-07-17
3,SPUTNIK 4 DEB,1960-005D,1961-03-15T15:48:42.802271,15.49800338,0.01984030,64.8899,343.2411,333.4201,27.0840,37,4690,0.00000000000000,0.00039007,0.0000000000000,6795.450,92.915,552.139,282.492,DEBRIS,None,None,None,1961-06-30
4,SPUTNIK 4 DEB,1960-005E,1960-08-19T16:34:33.722976,15.88909013,0.00342130,64.8899,302.1218,49.4823,310.6691,38,1500,0.00000000000000,0.00610805,0.0000000000000,6683.481,90.628,328.212,282.480,DEBRIS,None,None,None,1960-08-20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24620,CZ-6A DEB,2024-140XY,2024-12-11T23:05:15.774432,15.90490446,0.00187510,89.2070,338.9347,209.0449,150.9793,61407,1395,0.03161600000000,0.07121792,0.0914950000000,6679.046,90.538,313.435,288.388,DEBRIS,SMALL,2024-08-06,TSC,2024-12-17
24621,CZ-6A DEB,2024-140ZF,2024-11-13T06:57:45.539424,16.26800607,0.00303320,90.0847,353.4507,44.4626,315.9104,61438,1498,0.00486610000000,0.17891631,-0.0000005426100,6579.289,88.517,221.110,181.198,DEBRIS,SMALL,2024-08-06,TSC,2024-11-16
24622,CZ-2D R/B,2024-177B,2024-11-10T05:25:56.203680,16.37905997,0.00075290,41.3775,239.1072,234.7996,251.2891,61445,699,0.00067146000000,0.07484936,0.0000079825000,6549.516,87.917,176.312,166.450,ROCKET BODY,LARGE,2024-09-27,JSC,2024-11-10
24623,CZ-2F DEB,2024-194B,2024-11-22T23:07:29.851680,15.68300172,0.02578460,41.3237,193.5705,294.4302,62.9836,61684,378,0.00048903000000,0.00465987,0.0000065563000,6741.901,91.819,537.603,189.929,DEBRIS,LARGE,2024-10-29,JSC,2024-12-21


#### 3. Check the differences between Decay Date and Epoch (timestamp of OMM)

In [18]:
# check difference between Epoch and Decay Date
OMMs_df['EPOCH'] = pd.to_datetime(OMMs_df['EPOCH'])
OMMs_df['DECAY_DATE'] = pd.to_datetime(OMMs_df['DECAY_DATE'])
OMMs_df['Epoch_Decay_Difference'] = OMMs_df['DECAY_DATE'] - OMMs_df['EPOCH']
OMMs_df['Epoch_Decay_Difference'].describe()

count                          24625
mean      18 days 20:12:52.483248731
std      299 days 09:35:42.273035036
min            -10727 days +00:00:00
25%                  0 days 00:00:00
50%                  1 days 00:00:00
75%                  5 days 00:00:00
max               8300 days 00:00:00
Name: Epoch_Decay_Difference, dtype: object

In [19]:
# show objects where Epoch_Decay_Difference is negative
OMMs_df[OMMs_df['Epoch_Decay_Difference'] < pd.Timedelta(0)]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,LAUNCH_DATE,SITE,DECAY_DATE,Epoch_Decay_Difference
1,THOR ABLESTAR DEB,1960-003C,1960-10-06,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,33,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,1960-07-17,-81 days
2872,DELTA 1 DEB,1972-058C,2004-11-24,15.86540389,0.00128020,98.3597,332.5522,114.4864,245.7741,7832,57786,0.00141360000000,0.00371465,0.0000000000000,6690.132,90.763,320.561,303.432,DEBRIS,LARGE,1972-07-23,AFWTR,1977-02-12,-10147 days
3087,COSMOS 777 DEB,1975-102F,1976-05-02,15.50607889,0.00266690,65.0265,340.8773,6.7572,353.3878,8625,2876,0.00354340000000,0.00212521,0.0000000000000,6793.091,92.866,433.072,396.839,DEBRIS,None,None,None,1976-03-15,-48 days
3100,COSMOS 777 DEB,1975-102AE,1976-05-02,15.50641491,0.00104770,65.0344,341.0843,8.9752,351.1527,8650,2877,0.00000000000000,0.00246035,0.0000000000000,6792.993,92.864,421.975,407.741,DEBRIS,None,None,None,1976-03-27,-36 days
3142,COSMOS 777 DEB,1975-102BF,1976-05-02,15.40156937,0.00557380,64.8281,339.9362,23.9313,336.4347,8728,2872,0.00328330000000,0.00142121,0.0000000000000,6823.787,93.496,483.686,407.617,DEBRIS,None,None,None,1976-03-29,-34 days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23255,BION 11 DEB,1996-073G,1997-02-03,16.25788645,0.00087860,62.7868,87.1680,23.0743,337.0808,24710,439,0.00004231000000,0.00130341,0.0000086059000,6582.022,88.572,209.670,198.105,DEBRIS,SMALL,1996-12-24,PKMTR,1997-01-10,-24 days
23897,CZ-2F DEB,2024-078D,2024-04-28,16.31592083,0.00635030,41.5973,28.4800,152.0982,294.6586,59594,37,0.00245460000000,0.38721847,0.0000072850000,6566.402,88.257,229.965,146.568,DEBRIS,None,2024-04-25,JSC,2024-04-27,-1 days
24120,COSMOS 2251 DEB,1993-036AZQ,2024-05-11,14.67237796,0.00346710,73.9910,229.2548,89.3171,271.1971,36387,57450,0.02934800000000,0.00171058,0.0000000000000,7048.037,98.144,694.338,645.466,DEBRIS,SMALL,1993-06-16,PKMTR,2024-05-07,-4 days
24312,COSMOS 1408 DEB,1982-092CCH,2023-07-06,15.17775450,0.01267330,82.5778,267.2573,149.0686,211.8095,57287,1,0.00109990000000,0.00027530,0.0000000000000,6890.702,94.876,599.895,425.239,DEBRIS,None,None,None,2023-06-12,-24 days


In [23]:
# drop objects where OMM recorded after object was supposed to have decayed
OMMs_df = OMMs_df[OMMs_df['Epoch_Decay_Difference'] >= pd.Timedelta(0)]
print(f"Remaining objects: {len(OMMs_df)}")

# remove the difference column 
OMMs_df.drop(columns=['Epoch_Decay_Difference'], inplace=True)

Remaining objects: 24199


In [24]:
# Save OMMs to csv
OMMs_df.to_csv('../Data/DS_OMMs.csv', index=False)